# Python Iterators

Iterators and iterables form the core engine behind Python's loops, comprehensions, and data streaming pipelines. They allow programs to process sequences lazily and memory-efficiently.


### 1. Iterable vs. Iterator

An **Iterable** is any object that can return an iterator (it implements `__iter__()` or sequence `__getitem__()`).
Examples include lists, tuples, dictionaries, sets, and strings.

Iterable: has __iter__()

An **Iterator** is the stateful stream object that produces values one at a time via `__next__()`. It maintains an internal pointer to track its current position.

Iterator: has both __iter__() and __next__()


In [1]:
numbers_list = [10, 20, 30]  # Iterable

# Check methods using hasattr()
print("Is list an Iterable?", hasattr(numbers_list, "__iter__"))  # True
print("Is list an Iterator?", hasattr(numbers_list, "__next__"))  # False


Is list an Iterable? True
Is list an Iterator? False


In [2]:

# Obtain an iterator from the iterable
numbers_iter = iter(numbers_list)

print("\nIs numbers_iter an Iterable?", hasattr(numbers_iter, "__iter__"))  # True
print("Is numbers_iter an Iterator?", hasattr(numbers_iter, "__next__"))  # True
print("Type of iterator:", type(numbers_iter))


Is numbers_iter an Iterable? True
Is numbers_iter an Iterator? True
Type of iterator: <class 'list_iterator'>


### 2. The `iter()` Function

The built-in `iter(object)` returns an iterator for the given object. It also has a powerful **2-argument sentinel form**: `iter(callable, sentinel)`, which repeatedly calls the zero-argument callable until it returns the sentinel value, signaling the end.

**Syntax**:
```python
# Standard form:
it = iter(iterable)

# 2-argument sentinel form:
it = iter(callable, sentinel)
```

In [3]:
# Standard usage
greeting = "Hi!"
char_iter = iter(greeting)
print("Standard iter():", next(char_iter), next(char_iter), next(char_iter))

Standard iter(): H i !


In [5]:
# 2-argument sentinel form: iter(callable, sentinel)
# Great for reading chunks, random numbers, or socket streams until a specific marker
import random

def roll_die():
    return random.randint(1, 6)

print("\n--- Rolling die until we roll a 6 (sentinel) ---")
# Calls roll_die() repeatedly; stops immediately when roll_die() returns 6
die_roller = iter(roll_die, 6)
for roll in die_roller:
    print(f"Rolled: {roll}")
print("Sentinel (6) reached! Loop terminated.")


--- Rolling die until we roll a 6 (sentinel) ---
Rolled: 4
Rolled: 3
Rolled: 3
Rolled: 2
Rolled: 1
Rolled: 4
Rolled: 5
Rolled: 4
Rolled: 5
Rolled: 3
Sentinel (6) reached! Loop terminated.


### 3. The `next()` Function

Retrieves the next element from an iterator by invoking its `__next__()` method. If no elements remain, it raises `StopIteration`. You can supply an optional second argument `next(iterator, default)` to act as a fallback and avoid raising the exception.

**Syntax**:
```python
# Raises StopIteration when exhausted:
val = next(iterator)
# Safe fallback without exception:
val = next(iterator, default_value)
```

In [6]:
colors = ["red", "green"]
color_iter = iter(colors)

# Retrieving elements sequentially
print("1st call:", next(color_iter))  # 'red'
print("2nd call:", next(color_iter))  # 'green'

# Using default fallback to prevent StopIteration
print("3rd call (with default):", next(color_iter, "END_OF_STREAM"))
print("4th call (with default):", next(color_iter, "END_OF_STREAM"))


1st call: red
2nd call: green
3rd call (with default): END_OF_STREAM
4th call (with default): END_OF_STREAM


In [9]:

# Calling next() without default raises StopIteration
try:
    fresh_iter = iter(["hello"])
    print(next(fresh_iter)) # Prints hello
    print(next(fresh_iter)) # raises StopIteration
except StopIteration:
    print("\nCaught expected StopIteration on empty iterator!")

hello

Caught expected StopIteration on empty iterator!


### 4. The Iterator Protocol (`__iter__()` and `__next__()`)

Python's formal Iterator Protocol requires an object to implement two special (dunder) methods:

1. `__iter__()`: Returns the iterator object itself (`return self`). This enables iterators to be passed directly to `for` loops.
2. `__next__()`: Returns the next value in the stream, or raises `StopIteration` when the sequence is completed.

**Syntax**:
```python
class MyIterator:
    def __iter__(self):
        return self

    def __next__(self):
        if done:
            raise StopIteration
        return next_value
```

In [10]:
# 4. Verifying the Iterator Protocol
sample_data = ["Alpha", "Beta"]
it = iter(sample_data)

# __iter__() on an iterator returns the exact same object
print("Does it.__iter__() is it?", it.__iter__() is it)  # True

# Calling __next__() directly vs using next()
print("Calling it.__next__() :", it.__next__())
print("Calling next(it)      :", next(it))

Does it.__iter__() is it? True
Calling it.__next__() : Alpha
Calling next(it)      : Beta


### 5. How `for` Loops Work Internally

Under the hood, Python's `for item in iterable:` does not work by magic index lookups. It automatically calls `iter(iterable)` to obtain an iterator, repeatedly invokes `next(iterator)` inside an infinite loop, and cleanly exits when `StopIteration` is caught.

**Syntax**:
```python
# Conceptual equivalent of a for loop:
_iter = iter(iterable)
while True:
    try:
        item = next(_iter)
    except StopIteration:
        break
    # loop body
```

In [11]:
fruits = ["Apple", "Banana", "Cherry"]

print("--- Standard for loop ---")
for f in fruits:
    print(f)

--- Standard for loop ---
Apple
Banana
Cherry


In [12]:
print("\n--- Manual Emulation of for loop ---")
# 1. Fetch iterator from iterable
fruit_iterator = iter(fruits)

# 2. Infinite loop driving the iteration
while True:
    try:
        # 3. Retrieve next item
        item = next(fruit_iterator)
        # 4. Execute body
        print("Emulated:", item)
    except StopIteration:
        # 5. Cleanly terminate on exhaustion
        break


--- Manual Emulation of for loop ---
Emulated: Apple
Emulated: Banana
Emulated: Cherry


### 6. The `StopIteration` Exception

`StopIteration` is a built-in exception raised by an iterator's `__next__()` method to signal that no further items remain. Unlike errors, `StopIteration` is normal control flow in Python. Built-in constructs like `for`, `sum()`, `min()`, `max()`, and list comprehensions intercept it automatically.

**Syntax**:
```python
raise StopIteration('Iteration finished')
```

In [13]:
# 6. StopIteration Handling
nums = [1, 2]
it = iter(nums)

# Consume all elements
next(it)
next(it)

# Explicitly handling the termination signal
try:
    next(it)
except StopIteration as stop_err:
    print(f"StopIteration caught! Type: {type(stop_err).__name__}")
    print("Signaling all elements have been consumed.")

StopIteration caught! Type: StopIteration
Signaling all elements have been consumed.


### 7. Creating Custom Iterator Classes

You can build your own iterators by creating a class that implements both `__iter__()` and `__next__()`. Maintaining internal state (e.g. current index, step, or counter) allows iterating over custom mathematical sequences, ranges, or datasets.

**Syntax**:
```python
class Counter:
    def __init__(self, start, stop): ...
    
    def __iter__(self): return self

    def __next__(self): ...
```

In [14]:
# Example 1: Countdown Iterator
class Countdown:
    """Iterator that counts down from start to 1."""
    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        return self  # Iterator must return itself

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        val = self.current
        self.current -= 1
        return val

print("--- Custom Countdown Iterator ---")
for val in Countdown(5):
    print(val, end=" -> ")
print("Liftoff!\n")

--- Custom Countdown Iterator ---
5 -> 4 -> 3 -> 2 -> 1 -> Liftoff!



In [15]:
# Example 2: Bounded Fibonacci Iterator
class Fibonacci:
    """Generates Fibonacci numbers up to a maximum limit."""
    def __init__(self, max_limit: int):
        self.max_limit = max_limit
        self.a, self.b = 0, 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.a > self.max_limit:
            raise StopIteration
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        return result

print("--- Custom Fibonacci Iterator (limit 50) ---")
fib = Fibonacci(50)
print(list(fib))

--- Custom Fibonacci Iterator (limit 50) ---
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


---
## 8. Iterators vs. Sequences (Lists, Tuples, Strings)

| Feature | Sequences (`list`, `tuple`, `str`) | Iterators (`iter()`, generators) |
| :--- | :--- | :--- |
| **Memory Allocation** | Stores all elements in RAM simultaneously | Generates elements on-demand ($O(1)$ RAM) |
| **Length / Size** | Can check with `len(seq)` | No length; size is unknown until exhausted |
| **Indexing & Slicing** | Supported (`seq[0]`, `seq[1:4]`) | **Not supported** (`TypeError: not subscriptable`) |
| **Reusability** | Reusable indefinitely | **One-time use only**; consumed upon traversal |
| **Evaluation** | Eager (evaluated immediately upon creation) | Lazy (evaluated only when `next()` is called) |

In [19]:
# 8. Memory Footprint Comparison: List vs Iterator
import sys

# Generate 1,000,000 numbers eagerly in a list
large_list = list(range(1_000_000))

# Create an iterator over the same range
large_iter = iter(range(1_000_000))

print(f"Memory size of 1M element list: {sys.getsizeof(large_list):,} bytes")
print(f"Memory size of iterator       : {sys.getsizeof(large_iter):,} bytes")
print(f"Iterator is {sys.getsizeof(large_list) // sys.getsizeof(large_iter):,}x more memory-efficient!")

# Indexing comparison:
print("\nList supports indexing: large_list[5] =", large_list[5])
try:
    large_iter[5]
except TypeError as err:
    print("Iterator indexing fails:", err)

Memory size of 1M element list: 8,000,056 bytes
Memory size of iterator       : 40 bytes
Iterator is 200,001x more memory-efficient!

List supports indexing: large_list[5] = 5
Iterator indexing fails: 'range_iterator' object is not subscriptable


### 9. Built-in Iterators: `enumerate()`, `zip()`, `map()`, and `filter()`

Many of Python's built-in functional utilities do not create concrete lists; they return lazy, memory-efficient iterator objects that compute items as requested.

**Syntax**:
```text
enumerate(iterable, start=0) -> iterator

zip(*iterables) -> iterator

map(function, iterable) -> iterator

filter(function, iterable) -> iterator
```

In [20]:
names = ["Alice", "Bob", "Charlie"]
scores = [95, 88, 92]

# 1. enumerate() returns an enumerate iterator
enum_it = enumerate(names, start=1)
print("Type of enumerate:", type(enum_it))
print("Next from enumerate:", next(enum_it))  # (1, 'Alice')

Type of enumerate: <class 'enumerate'>
Next from enumerate: (1, 'Alice')


In [21]:
# 2. zip() returns a zip iterator
zip_it = zip(names, scores)
print("\nType of zip:", type(zip_it))
print("Next from zip:", next(zip_it))  # ('Alice', 95)


Type of zip: <class 'zip'>
Next from zip: ('Alice', 95)


In [22]:
# 3. map() returns a map iterator (lazy transformation)
sq_it = map(lambda x: x ** 2, [1, 2, 3, 4])
print("\nType of map:", type(sq_it))
print("Consuming map:", list(sq_it))


Type of map: <class 'map'>
Consuming map: [1, 4, 9, 16]


In [25]:
# 4. filter() returns a filter iterator (lazy predicate test)
even_it = filter(lambda x: x % 2 == 0, [10, 15, 20, 25])
print("\nType of filter:", type(even_it))
print("Consuming filter:", list(even_it))


Type of filter: <class 'filter'>
Consuming filter: [10, 20]


### 10. Infinite Iterators (`itertools` Module)

Because iterators compute elements on the fly rather than storing them in memory, they can represent infinitely long streams without running out of RAM. The `itertools` module provides `count()`, `cycle()`, and `repeat()`.

In [27]:
import itertools

# 1. itertools.count(start, step): Infinite numerical counter
print("--- itertools.count(start=100, step=10) ---")
counter = itertools.count(start=100, step=10)
for _ in range(5):
    print(next(counter), end=" ")
print()

--- itertools.count(start=100, step=10) ---
100 110 120 130 140 


In [28]:
# 2. itertools.cycle(iterable): Infinitely cycles through a sequence
print("\n--- itertools.cycle(['RED', 'YELLOW', 'GREEN']) ---")
traffic_lights = itertools.cycle(["RED", "YELLOW", "GREEN"])
for _ in range(7):
    print(next(traffic_lights), end=" -> ")
print()


--- itertools.cycle(['RED', 'YELLOW', 'GREEN']) ---
RED -> YELLOW -> GREEN -> RED -> YELLOW -> GREEN -> RED -> 


In [29]:
# 3. itertools.repeat(elem, n): Repeats an element
print("\n--- itertools.repeat('PING', 4) ---")
pings = itertools.repeat("PING", 4)
print(list(pings))


--- itertools.repeat('PING', 4) ---
['PING', 'PING', 'PING', 'PING']


### 11. Iterator Exhaustion & One-Time Consumption

A critical property of all iterators: **they can only be consumed once**. Once an iterator raises `StopIteration` or runs to completion, its internal pointer remains at the end. Subsequent loops or `next()` calls will yield nothing. To re-iterate, you must instantiate a new iterator.

**Syntax**:
```python
it = iter(collection)

list(it)  # Consumed!
list(it)  # Returns empty []
```

In [30]:
# 11. Iterator Exhaustion Demonstration
items = [1, 2, 3]
my_iter = iter(items)

print("First pass consuming iterator:")
for x in my_iter:
    print("Item:", x)

print("\nSecond pass with the SAME iterator:")
second_pass_items = list(my_iter)
print("Items collected:", second_pass_items)  # Empty list!

# Solution: Re-create the iterator from the iterable
fresh_iter = iter(items)
print("Fresh iterator items:", list(fresh_iter))

First pass consuming iterator:
Item: 1
Item: 2
Item: 3

Second pass with the SAME iterator:
Items collected: []
Fresh iterator items: [1, 2, 3]


### 12. Practical Real-World Use Cases

Iterators shine in real-world scenarios requiring scalable data processing: streaming large log files, paginating through API responses, and processing database record batches in fixed chunks without holding entire datasets in memory.

Examples: Processing infinite streams, paginated APIs, or batching datasets

In [31]:
# Use Case 1: Chunked / Batched Iterator for Datasets
class BatchIterator:
    """Yields data in chunks/batches of fixed size (e.g. for ML training or DB bulk inserts)."""
    def __init__(self, data_source: list, batch_size: int):
        self.data_source = data_source
        self.batch_size = batch_size
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.data_source):
            raise StopIteration
        batch = self.data_source[self.index : self.index + self.batch_size]
        self.index += self.batch_size
        return batch

dataset = list(range(1, 21))  # 20 records
print("--- Processing dataset in batches of 5 ---")
for batch_num, batch in enumerate(BatchIterator(dataset, batch_size=5), start=1):
    print(f"Batch {batch_num}: {batch}")

--- Processing dataset in batches of 5 ---
Batch 1: [1, 2, 3, 4, 5]
Batch 2: [6, 7, 8, 9, 10]
Batch 3: [11, 12, 13, 14, 15]
Batch 4: [16, 17, 18, 19, 20]


In [33]:
# Use Case 2: Paged API Iterator
class MockAPIPaginator:
    """Simulates paginated REST API fetching page by page until has_more is False."""
    def __init__(self, total_pages: int):
        self.total_pages = total_pages
        self.current_page = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_page > self.total_pages:
            raise StopIteration
        page_data = {
            "page": self.current_page,
            "records": [f"user_{self.current_page}_{i}" for i in range(1, 4)],
            "has_more": self.current_page < self.total_pages
        }
        self.current_page += 1
        return page_data

print("\n--- Streaming Paginated API Results ---")
api_stream = MockAPIPaginator(total_pages=3)
for response in api_stream:
    print(f"Page {response['page']} records:", response['records'])


--- Streaming Paginated API Results ---
Page 1 records: ['user_1_1', 'user_1_2', 'user_1_3']
Page 2 records: ['user_2_1', 'user_2_2', 'user_2_3']
Page 3 records: ['user_3_1', 'user_3_2', 'user_3_3']
